# Phase 5 prime replay-observed context gate (Colab)

This notebook runs the narrow Phase 5 prime context-source gate only. It is not a Phase 5 graduation run, not a full matrix, not an M2 path, and not an MQAR/bAbI headline pivot.

Target cell: `D=4096`, `K_roles=16`, `N=512`, `cue_noise=0.15`, `scene_token_source=replay_observed_context_trace`, `context_roles=4`, `scene_token_weight=0.25`, `cooccurrence=skewed`.

Required controls: `candidate`, `random_role`, `deranged_role`, `shuffled_role`, `content_cleanup_positive`.

The passive trace snapshot must contain `pattern_encoder_terms`. If there is not enough support for the hard cell, stop and report the support deficit instead of changing the experiment into a matrix or falling back to direct synthetic context.


In [ ]:
# 1. Clone the repo and check out the approved branch.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git /content/Neuro-AI
%cd /content/Neuro-AI
!git checkout codex/phase5-prime-replay-context-source
!git log --oneline -3

# Verify the passive trace source is present.
!grep -n "replay_observed_context_trace" experiments/44_phase5_prime_bundle_first.py | head -10


In [ ]:
# 2. Mount Drive and locate the provenance-bearing Phase 3/4 snapshot.
# If Drive auth is requested, approve it in the Colab UI.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

SNAPSHOT_NAME = 'phase3_phase4_w4_step1800.pt'
likely_paths = [
    Path('/content/drive/MyDrive/neuro-ai/reports/phase5_m1_provenance_seed17/snapshots') / SNAPSHOT_NAME,
    Path('/content/drive/MyDrive/Neuro-AI/reports/phase5_m1_provenance_seed17/snapshots') / SNAPSHOT_NAME,
    Path('/content/drive/MyDrive/results/phase5_m1_provenance_seed17/snapshots') / SNAPSHOT_NAME,
    Path('/content/drive/MyDrive/phase5_m1_provenance_seed17/snapshots') / SNAPSHOT_NAME,
]

matches = [p for p in likely_paths if p.is_file()]
if not matches:
    root = Path('/content/drive/MyDrive')
    matches = list(root.rglob(SNAPSHOT_NAME)) if root.exists() else []

print('snapshot matches:')
for p in matches[:20]:
    print(' ', p)

if not matches:
    raise FileNotFoundError(
        'No Drive snapshot named phase3_phase4_w4_step1800.pt was found. '
        'Upload or copy the provenance-bearing snapshot, then rerun this cell.'
    )

SNAPSHOT = str(matches[0])
print('using SNAPSHOT =', SNAPSHOT)


In [ ]:
# 3. Probe passive trace support before running the experiment.
import torch

state = torch.load(SNAPSHOT, map_location='cpu', weights_only=False)
rows = state.get('pattern_encoder_terms')
if rows is None:
    raise ValueError('snapshot has no pattern_encoder_terms')

def support_for(C_codebook, K_roles=16, context_roles=4):
    eligible = 0
    too_short = 0
    invalid = 0
    max_role = -1
    max_atom = -1
    for raw_terms in rows:
        if raw_terms is None:
            too_short += 1
            continue
        role_to_atom = {}
        row_invalid = False
        for raw_role, raw_atom in raw_terms:
            role = int(raw_role)
            atom = int(raw_atom)
            max_role = max(max_role, role)
            max_atom = max(max_atom, atom)
            if role < 0 or role >= K_roles or atom < 0 or atom >= C_codebook:
                row_invalid = True
                continue
            role_to_atom.setdefault(role, atom)
        if row_invalid:
            invalid += 1
        if len(role_to_atom) < context_roles:
            too_short += 1
            continue
        eligible += 1
    return {
        'C_codebook': C_codebook,
        'raw_rows': len(rows),
        'eligible_rows': eligible,
        'rows_too_short': too_short,
        'rows_invalid': invalid,
        'max_role_seen': max_role,
        'max_atom_seen': max_atom,
    }

support_rows = [support_for(C) for C in [1024, 2048, 4096, 8192]]
for item in support_rows:
    print(item)

C_CODEBOOK = next((item['C_codebook'] for item in support_rows if item['eligible_rows'] >= 512), None)
if C_CODEBOOK is None:
    raise ValueError('passive trace support deficit: no probed C_codebook yields 512 eligible rows')
print('using C_CODEBOOK =', C_CODEBOOK)


In [ ]:
# 4. Tiny Colab smoke for wiring only. This is not evidence for the hard cell.
import json, os, subprocess
from pathlib import Path

env = dict(os.environ, PYTHONPATH='src:.')
subprocess.run(['python', '-m', 'py_compile', 'experiments/44_phase5_prime_bundle_first.py'], check=True, env=env)

smoke_out = '/content/phase5_prime_replay_observed_context_smoke.json'
smoke_cmd = [
    'python', 'experiments/44_phase5_prime_bundle_first.py',
    '--Ds', '128',
    '--K_roles', '6',
    '--Ns', '8',
    '--cue_noise', '0.0',
    '--seeds', '17',
    '--n_queries', '16',
    '--C_codebook', str(C_CODEBOOK),
    '--scene_token', '1',
    '--scene_token_weight', '0.25',
    '--scene_token_source', 'replay_observed_context_trace',
    '--context_roles', '2',
    '--context_trace_snapshot', SNAPSHOT,
    '--conditions', 'candidate', 'random_role', 'deranged_role', 'shuffled_role', 'content_cleanup_positive',
    '--cooccurrence', 'skewed',
    '--device', 'cpu',
    '--out', smoke_out,
]
subprocess.run(smoke_cmd, check=True, env=env)
smoke = json.loads(Path(smoke_out).read_text())
print(json.dumps(smoke['framing'], indent=2))
for key, agg in sorted(smoke['aggregates'].items()):
    print(key, 'top1=', round(agg['top1_mean'], 4), 'support=', agg['passive_trace_rows_available'], agg['passive_trace_rows_used'])


In [ ]:
# 5. Single hard-cell gate. Do not expand this into a matrix in this notebook.
hard_out = '/content/phase5_prime_replay_observed_context_hard_cell.json'
hard_cmd = [
    'python', 'experiments/44_phase5_prime_bundle_first.py',
    '--Ds', '4096',
    '--K_roles', '16',
    '--Ns', '512',
    '--cue_noise', '0.15',
    '--seeds', '17', '11', '23', '1', '2', '3', '5', '7', '13', '29',
    '--n_queries', '512',
    '--C_codebook', str(C_CODEBOOK),
    '--scene_token', '1',
    '--scene_token_weight', '0.25',
    '--scene_token_source', 'replay_observed_context_trace',
    '--context_roles', '4',
    '--context_trace_snapshot', SNAPSHOT,
    '--conditions', 'candidate', 'random_role', 'deranged_role', 'shuffled_role', 'content_cleanup_positive',
    '--cooccurrence', 'skewed',
    '--device', 'auto',
    '--out', hard_out,
]
subprocess.run(hard_cmd, check=True, env=env)
hard = json.loads(Path(hard_out).read_text())
for key, agg in sorted(hard['aggregates'].items()):
    print(
        f"{key}\n"
        f"  top1={agg['top1_mean']:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] "
        f"scene_tix={agg['scene_tix_rate']:.4f} content_tix={agg['content_tix_rate']:.4f}\n"
        f"  support_available={agg['passive_trace_rows_available']} support_used={agg['passive_trace_rows_used']} "
        f"invalid={agg['passive_trace_rows_invalid']} too_short={agg['passive_trace_rows_too_short']}"
    )

print('\nWrote', hard_out)
print('Boundary: diagnostic only; no Phase 5 graduation, no Delta E headline, no full matrix, no M2 commitment.')


In [ ]:
# 6. Optional download of the two result JSONs from the Colab runtime.
from google.colab import files
for path in ['/content/phase5_prime_replay_observed_context_smoke.json', '/content/phase5_prime_replay_observed_context_hard_cell.json']:
    if Path(path).exists():
        print('downloading', path)
        files.download(path)
